In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_california_housing

# 1. Фіксуємо сід для відтворюваності
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 2. Завантаження даних California Housing
data = fetch_california_housing(as_frame=True)
df = data.frame

print("--- ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО ДАНІ ---")
print(f"Розмір набору даних: {df.shape}")
print(df.info())

print("\n--- СТАТИСТИКА ТА ПРОПУСКИ ---")
print(df.isnull().sum())
print(df.describe())

# 3. Етап кластеризації для створення «регіонів» (GroupKFold вимагає груп)
kmeans = KMeans(n_clusters=15, random_state=RANDOM_STATE, n_init=10)
df["RegionGroup"] = kmeans.fit_predict(df[["Latitude", "Longitude"]])

print(f"\nКількість сформованих регіональних груп: {df['RegionGroup'].nunique()}")
print(df.head(3))


Кількість сформованих регіональних груп: 15
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   

   Longitude  MedHouseVal  RegionGroup  
0    -122.23        4.526            1  
1    -122.22        3.585            1  
2    -122.24        3.521            1  


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor

# Відокремлюємо ознаки, ціль та групи
X = df.drop(columns=["MedHouseVal", "RegionGroup"])
y = df["MedHouseVal"]
groups = df["RegionGroup"]

# Визначаємо моделі для порівняння (з пасткою від витоку даних — StandardScaler всередині Pipeline!)
models = {
    "Dummy (Baseline)": DummyRegressor(strategy="mean"),
    "Linear Regression": Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())]),
    "Ridge": Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}

print("Моделі та конвеєри успішно ініціалізовані!")

Моделі та конвеєри успішно ініціалізовані!


In [3]:
# Етап 3: Порівняння випадкової крос-валідації (KFold) та групової за регіонами (GroupKFold)

scoring = {
    'RMSE': 'neg_root_mean_squared_error',
    'MAE': 'neg_mean_absolute_error',
    'R2': 'r2'
}

results_kfold = []
results_group = []

# 1. Звичайна випадкова крос-валідація (KFold)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    cv_res = cross_validate(model, X, y, cv=kf, scoring=scoring, n_jobs=-1)
    results_kfold.append({
        'Model': name,
        'Scheme': 'KFold (Random)',
        'RMSE_mean': -cv_res['test_RMSE'].mean(),
        'RMSE_std': cv_res['test_RMSE'].std(),
        'MAE_mean': -cv_res['test_MAE'].mean(),
        'MAE_std': cv_res['test_MAE'].std(),
        'R2_mean': cv_res['test_R2'].mean(),
        'R2_std': cv_res['test_R2'].std()
    })

# 2. Групова крос-валідація за регіонами (GroupKFold)
gkf = GroupKFold(n_splits=5)

for name, model in models.items():
    cv_res = cross_validate(model, X, y, groups=groups, cv=gkf, scoring=scoring, n_jobs=-1)
    results_group.append({
        'Model': name,
        'Scheme': 'GroupKFold (Spatial)',
        'RMSE_mean': -cv_res['test_RMSE'].mean(),
        'RMSE_std': cv_res['test_RMSE'].std(),
        'MAE_mean': -cv_res['test_MAE'].mean(),
        'MAE_std': cv_res['test_MAE'].std(),
        'R2_mean': cv_res['test_R2'].mean(),
        'R2_std': cv_res['test_R2'].std()
    })

# Об'єднуємо результати в єдину таблицю
df_results = pd.DataFrame(results_kfold + results_group)
print("--- ПОРІВНЯННЯ РЕЗУЛЬТАТІВ КРОС-ВАЛІДАЦІЇ ---")
print(df_results.to_string(index=False))

# Зберігаємо результати у папку results для звіту
import os
os.makedirs('results', exist_ok=True)
df_results.to_csv('results/model_comparison.csv', index=False)
print("\nРезультати успішно збережено у 'results/model_comparison.csv'!")

--- ПОРІВНЯННЯ РЕЗУЛЬТАТІВ КРОС-ВАЛІДАЦІЇ ---
               Model               Scheme  RMSE_mean  RMSE_std  MAE_mean  MAE_std   R2_mean   R2_std
    Dummy (Baseline)       KFold (Random)   1.153941  0.010349  0.911758 0.006706 -0.000282 0.000348
   Linear Regression       KFold (Random)   0.728251  0.014936  0.531677 0.008432  0.601378 0.017010
               Ridge       KFold (Random)   0.728249  0.014928  0.531672 0.008432  0.601381 0.017001
       Random Forest       KFold (Random)   0.506457  0.008044  0.329917 0.006517  0.807252 0.006136
HistGradientBoosting       KFold (Random)   0.466455  0.010266  0.311750 0.005770  0.836427 0.007930
    Dummy (Baseline) GroupKFold (Spatial)   1.177364  0.110601  0.941161 0.118502 -0.168779 0.279016
   Linear Regression GroupKFold (Spatial)   0.737867  0.093736  0.545997 0.065849  0.549503 0.068152
               Ridge GroupKFold (Spatial)   0.737845  0.093791  0.545978 0.065885  0.549542 0.068109
       Random Forest GroupKFold (Spatial)   0

In [5]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np

# Беремо найкращу модель за результатами минулого етапу
best_model = models["HistGradientBoosting"]

# 1. Permutation Importance для KFold (випадковий поділ)
# Навчаємо на всій вибірці для демонстрації важливості (або на фолдах)
best_model.fit(X, y)
result_kfold = permutation_importance(
    best_model, X, y, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

# 2. Побудова графіка для permutation importance
sorted_idx_kfold = result_kfold.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.barh(X.columns[sorted_idx_kfold], result_kfold.importances_mean[sorted_idx_kfold])
plt.xlabel("Permutation Importance (Mean Decrease in Score)")
plt.title("Permutation Importance (HistGradientBoosting)")
plt.tight_layout()

# Зберігаємо графік у папку figures
import os
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/permutation_importance.png")
plt.show()

print("Permutation importance успішно розраховано, а графік збережено у 'figures/permutation_importance.png'!")

Permutation importance успішно розраховано, а графік збережено у 'figures/permutation_importance.png'!


In [6]:
# Етап 5: Аналіз помилок (Residuals)
residuals = y - best_model.predict(X)
df_errors = X.copy()
df_errors['Actual'] = y
df_errors['Predicted'] = best_model.predict(X)
df_errors['Residual'] = residuals
df_errors['AbsError'] = np.abs(residuals)
df_errors['RegionGroup'] = groups

# Виводимо топ-5 найбільших помилок моделі
print("--- ТОП-5 НАЙБІЛЬШИХ ПОМИЛОК МОДЕЛІ ---")
print(df_errors.sort_values(by='AbsError', ascending=False)[['MedInc', 'HouseAge', 'Actual', 'Predicted', 'AbsError']].head(5))

# Візуалізація помилок на географічній карті (Latitude vs Longitude, колір за величиною помилки)
plt.figure(figsize=(10, 6))
sc = plt.scatter(df_errors['Longitude'], df_errors['Latitude'], c=df_errors['AbsError'], cmap='hot', alpha=0.6, s=15)
plt.colorbar(sc, label='Absolute Error')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Просторові помилки моделі (Absolute Residuals на карті)')
plt.tight_layout()

os.makedirs("figures", exist_ok=True)
plt.savefig("figures/spatial_errors.png")
plt.show()

print("Аналіз помилок завершено, графік збережено у 'figures/spatial_errors.png'!")

Аналіз помилок завершено, графік збережено у 'figures/spatial_errors.png'!
